# GLiNER large -- helio-pii multi-label eval

Loads `urchade/gliner_large-v2.1`, drives it with the `configs/taxonomy.json`
zero-shot prompts, applies each canonical label's own threshold from
`configs/thresholds.json`, and scores it against the
`data/test_harness.jsonl` multi-label classification harness.

In [ ]:
import json
import sys

import pandas as pd
from gliner import GLiNER

sys.path.append("../src")
from taxonomy import load_taxonomy
from metrics import evaluate

taxonomy = load_taxonomy("../configs/taxonomy.json", "../configs/thresholds.json")
taxonomy.thresholds

In [ ]:
model = GLiNER.from_pretrained("urchade/gliner_large-v2.1")

In [ ]:
with open("../data/test_harness.jsonl") as f:
    docs = [json.loads(line) for line in f]
len(docs)

In [ ]:
def predict_spans(text: str) -> list[dict]:
    """Raw span hits with each prompt's own canonical label attached, for
    eyeballing what actually fired. GLiNER's labels ARE the prompt strings,
    so multiple prompt_labels can fire on the same canonical category."""
    raw = model.predict_entities(text, taxonomy.all_prompts, threshold=taxonomy.score_floor, multi_label=True)
    for e in raw:
        e["canonical"] = taxonomy.prompt_to_canonical[e["label"]]
    return raw


def predict_canonical_labels(text: str) -> list[str]:
    """Doc-level prediction: max prompt score per canonical label, then that
    label's own threshold. A canonical label with no span clearing its
    threshold is simply absent -- that's the "non-PII" case, no extra work."""
    best_per_canonical: dict[str, float] = {}
    for e in predict_spans(text):
        canonical = e["canonical"]
        if e["score"] > best_per_canonical.get(canonical, -1.0):
            best_per_canonical[canonical] = e["score"]
    return [c for c, score in best_per_canonical.items() if score >= taxonomy.threshold_for(c)]

In [ ]:
predictions = [predict_canonical_labels(d["text"]) for d in docs]
for d, pred in zip(docs, predictions):
    print(f"{d['id']:24s} gold={d['labels']!s:30s} pred={pred}")

In [ ]:
gold_label_sets = [d["labels"] for d in docs]
rows, micro, macro = evaluate(gold_label_sets, predictions, taxonomy.canonical_names)

per_label_df = pd.DataFrame(rows).sort_values("label").reset_index(drop=True)
per_label_df

In [ ]:
print("micro:", micro)  # dominated by frequent labels
print("macro:", macro)  # every label weighted equally -- watch this for rare PHI labels

In [ ]:
# The harness tags each row with scenario_type/noise_type/polarity -- break
# exact-match accuracy down by scenario_type to see where it's weakest
# (e.g. boundary_ambiguous vs. near_miss_negative).
harness_df = pd.DataFrame(docs)
harness_df["predicted"] = predictions
harness_df["exact_match"] = [set(g) == set(p) for g, p in zip(gold_label_sets, predictions)]
harness_df.groupby("scenario_type")["exact_match"].mean()